In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import networkx as nx
from itertools import permutations, combinations
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 600  # keep the same as dpi setting in figure.savefig
import matplotlib.lines as mlines
import matplotlib.dates as mdates
from shapely.geometry import Point

In [ ]:
#1. load study_area, regionalization and specify node locations on the map
study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")   

In [ ]:
ctr_pos = dict()

for index, row in study_area.iterrows():
    ctr_name = row['ISO3_CODE']
    
    ctr_geometry = gpd.GeoSeries(data = [row.geometry], crs = "EPSG:4326")
    ctr_geometry = ctr_geometry.to_crs(epsg = 3035)
    
    #get fire centroid in 3035 prj
    ctr_centroid = zip(ctr_geometry.geometry.centroid.x, ctr_geometry.geometry.centroid.y)

    #reproject the centroid to 4326 prj
    gdf_ctr_centroid = gpd.GeoDataFrame({'geometry': [Point(ctr_centroid)]}, crs = "EPSG:3035")
    gdf_ctr_centroid = gdf_ctr_centroid.to_crs(epsg = 4326)

    ctr_pos.update({ctr_name: (gdf_ctr_centroid.iloc[0].geometry.x, gdf_ctr_centroid.iloc[0].geometry.y)})

In [ ]:
#plot country centroids
fig, ax = plt.subplots(1, 1, figsize = (5, 8))
study_area.boundary.plot(ax = ax, linewidth = 0.1, color = 'blue')

for ctr_name, pos in ctr_pos.items():
    ax.scatter(pos[0], pos[1], color = 'red', label = 'ctr_name')

In [ ]:
ctr_pos.update({'ESP':(-3, 38),
                'HRV': (16, 45.5),
                'GBR': (-3.3, 55.5)})  #adjust node positions for better visibility

In [ ]:
fig, axes = plt.subplots(2, 3, figsize = (19, 18))
axes = axes.flatten()

season_dict = dict(zip([0, 1, 2, 3, 4], ["all_seasons", "MAM", "JJA", "SON", "DJF"]))
title_dict = dict(zip([0, 1, 2, 3, 4], ["(a) All seasons", "(b) MAM", "(c) JJA", "(d) SON", "(e) DJF"]))
tau_max = 6
scaling_factor = 0.6 # Adjust this factor for the thickness of the link

ctr_namelist = list(ctr_pos.keys())

for ax_id, ax in enumerate(axes):

    #legend panel--------------------------------------------------------------------------------------------------
    if ax_id == 5:
        
        ax.axis("off")

        # create the legend for log(ES)
        ES_linewidth = [np.log(es) * scaling_factor for es in [10, 25, 50, 100, 200, 300]]
        ES_label = [10, 25, 50, 100, 200, 300]
    
        
        legend3 = ax.legend(handles=[mlines.Line2D([], [], color = 'blue', alpha = 0.2, linewidth = lw, label = f"{lb:<4}",  # left-aligned, 4-character width
                                                     ) for lb, lw in zip(ES_label, ES_linewidth)],
                            loc = 'upper left', fontsize = 17, title = 'Event synchronicity (-)', title_fontsize = 17, frameon = False, bbox_to_anchor = (0.2, 0.65), ncol = 1, handletextpad = 0.3)
        

    #plot panel----------------------------------------------------------------------------------------------------
    else:
        
        # get season indicator
        season = season_dict[ax_id]

        
        # load event synchronicity matrix
        ES = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Event_Sync_country_level/ES_matrix_country-level_taumax_{tau_max}_{season}.csv", index_col = 0)
    
        # aggregate the matrix to upper right
        for i in range(len(ES)):
            for j in range(i, len(ES)):
                if i == j:
                    ES.iloc[i,j] = 0
                else:
                    ES.iloc[i,j] += ES.iloc[j,i]
                    ES.iloc[j,i] = 0
                
        #--------------------------------------------------------------------------------------
        #construct networks for all links
        G = nx.Graph()
        
        #add nodes
        G.add_nodes_from(ctr_namelist)
        
        #add edges
        for region_combo in combinations(ctr_namelist, 2):
            syn = ES.loc[region_combo[0], region_combo[1]]
            G.add_edge(region_combo[0], region_combo[1], weight=syn)
                
        #edge weight
        weights = nx.get_edge_attributes(G, 'weight')

        if season == 'all_seasons':  #only show ES >= 20
            
            scaled_weights = [np.log(weight) if weight >= 20 else 0 for weight in weights.values()]

        else:  #only show ES >= 10

            scaled_weights = [np.log(weight) if weight >= 10 else 0 for weight in weights.values()]
            
        edge_widths = [w * scaling_factor for w in scaled_weights]
        
    
        #--------------------------------------------------------------------------------------
        #Draw the study area and regionalization
        study_area.boundary.plot(ax = ax, color = 'black', linewidth = 0.5, zorder = 0)
        
        #Draw the network
        # Draw edges 
        nx.draw_networkx_edges(G, pos = ctr_pos, ax=ax, edge_color="blue", width=edge_widths, alpha = 0.2).set_zorder(1)

        
        # Draw nodes
        nx.draw_networkx_nodes(G, pos = ctr_pos, ax=ax, node_color='red', node_size=10).set_zorder(3)

        # subplot label
        ax.text(0.04, 0.92, title_dict[ax_id], transform = ax.transAxes, fontsize = 20)
        
        # turn the axis on
        ax.set_axis_on()
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
        ax.set_xlabel("")
        ax.set_ylabel("")
        #--------------------------------------------------------------------------------------


plt.subplots_adjust(wspace = 0.04, hspace = 0.02) 
#fig.savefig(f"/home/lixinh/SynFire_Scripts/WP1/Figures/R_CEE_SynMap_country_level.png", dpi = 600, bbox_inches = "tight")
plt.show()